In [ ]:
!pip install xgboost

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score,mean_absolute_percentage_error
from xgboost import XGBRegressor

# 1. Load data
df = pd.read_csv(r"C:\Users\Saravanan\Desktop\demo\e25764c65bca11f0\dataset\train.csv")
df.head()


X = df.iloc[:, :55]    # component fraction + 10 properties
y = df.iloc[:, 55:]    # blend properties
y = y.apply(pd.to_numeric, errors='coerce').fillna(0)

# 3. Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 4. Train-validation split
X_train, X_val, y_train, y_val = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

# 5. Train model with XGBoost
xgb_model = MultiOutputRegressor(
    XGBRegressor(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=8,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1
    )
)
xgb_model.fit(X_train, y_train)

# 6. Validation predictions
y_pred_val = xgb_model.predict(X_val)

# 7. Evaluation Metrics
rmse = np.sqrt(mean_squared_error(y_val, y_pred_val))
mae = mean_absolute_error(y_val, y_pred_val)
r2 = r2_score(y_val, y_pred_val)
mape = mean_absolute_percentage_error(y_val, y_pred_val)

print("===== Overall Validation Performance (XGBoost) =====")
print(f"RMSE : {rmse:.4f}")
print(f"MAE  : {mae:.4f}")
print(f"R²   : {r2:.4f}")
print(f"MAPE : {mape*100:.2f}%")

print("\n===== Per-Property Performance =====")
for i, col in enumerate(y.columns):
    rmse_i = np.sqrt(mean_squared_error(y_val.iloc[:, i], y_pred_val[:, i]))
    mae_i = mean_absolute_error(y_val.iloc[:, i], y_pred_val[:, i])
    r2_i = r2_score(y_val.iloc[:, i], y_pred_val[:, i])
    mape_i = mean_absolute_percentage_error(y_val.iloc[:, i], y_pred_val[:, i])
    print(f"{col}: RMSE={rmse_i:.4f}, MAE={mae_i:.4f}, R²={r2_i:.4f}, MAPE={mape_i*100:.2f}%")

# 8. Retrain on full data & predict test
xgb_model.fit(X_scaled, y)
y_test_pred = xgb_model.predict(X_val)





